# dbt-for-apache-doris

## 5 个端到端数据工程 Demo

从 Doris Source 开始，依次运行 dbt model、Data Test 和 verifier。5 个 Demo 都按数据转换过程拆成可依次执行的单元格，每一步展示输入文件、dbt SQL、Doris 中间结果或校验结果。每个 Demo 使用独立 Database；完整原始日志默认折叠，排错时再展开。

| Demo | 主要能力 | 最终对象 |
| --- | --- | --- |
| 每日订单汇总 | Table、Data Test、分区分桶、Async MV | 每日和月度收入 |
| 客户地域分析 | 跨 Database Source、View、`ref()` | 州级客户和收入指标 |
| 广告数据合并 | Seed、`dbt_utils`、`QUALIFY` | 三渠道统一明细 |
| 迟到订单 | Incremental `merge`、Unique Key | 去重后的订单当前版本 |
| 客户 Snapshot | SCD Type 2、Hard Delete | 客户历史和当前维表 |

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "extension/dbt-doris/examples/data-eng-bench-doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 Apache Doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 1：每日订单汇总

这个 Demo 拆成 6 个依次执行的单元格。先运行环境检查，再按 2.1 到 2.6 的顺序执行；每一步都会展示输入或 dbt 文件、运行命令和对应的 Doris 结果。

```text
6 条源订单
  └─ 过滤 CANCELLED / RETURNED / FAILED
       └─ 3 条有效订单
            └─ 按日期聚合：daily_order_summary（3 行）
                 └─ 按月份聚合：monthly_order_summary_mv（1 行）
```

### 2.1 准备并查看源订单

Fixture 创建 6 条订单。最后一列直接标出每条记录会进入模型还是会被过滤。

In [ ]:
daily_demo_dir = runner.examples_root / "data-eng-bench-daily-order-summary"
runner.show_file("Fixture SQL", daily_demo_dir / "scripts/setup.sql")
runner.run_sql_file("创建源订单", daily_demo_dir / "scripts/setup.sql")
runner.query("输入：6 条原始订单", """
select
    order_id, ordered_at, grand_total, status,
    case
        when status in ('CANCELLED', 'RETURNED', 'FAILED') then '过滤'
        else '进入每日汇总'
    end as transform_action
from dbt_demo_daily_source.orders
order by order_id
""")

### 2.2 将 Doris 表声明为 dbt Source

`sources.yml` 把 `source('orders', 'orders')` 映射到 Doris 的 `dbt_demo_daily_source.orders`。`dbt debug` 随后检查这套 profile 能否连接 Doris。

In [ ]:
runner.show_file("dbt Source 配置", daily_demo_dir / "models/staging/sources.yml")
runner.run_dbt("检查 Demo 的 Doris 连接", daily_demo_dir, "debug")

### 2.3 执行每日汇总 Model

Model 读取 Source，过滤 3 种无效状态，然后按 `order_date` 聚合。dbt-doris 根据 `config()` 创建带 Range Partition、Duplicate Key 和 Hash Bucket 的 Doris Table。

In [ ]:
runner.show_file("每日汇总 Model", daily_demo_dir / "models/marts/daily_order_summary.sql")
runner.run_dbt("创建 daily_order_summary", daily_demo_dir, "run", "--select", "daily_order_summary")
runner.query("输出：3 天的有效订单汇总", """
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date
""")

### 2.4 执行 Data Test

这一步不再转换数据，而是检查每日结果：日期必须非空且唯一，订单数和收入必须非空。

In [ ]:
runner.show_file("Data Test 定义", daily_demo_dir / "models/marts/daily_order_summary.yml")
runner.run_dbt("验证 daily_order_summary", daily_demo_dir, "test", "--select", "daily_order_summary")

### 2.5 从每日表生成月度异步物化视图

第二个 Model 不再读取源订单，而是通过 `ref('daily_order_summary')` 读取上一步的 3 行结果，再按月份聚合。

In [ ]:
runner.show_file("月度物化视图 Model", daily_demo_dir / "models/marts/monthly_order_summary_mv.sql")
runner.run_dbt("创建月度异步物化视图", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.run_dbt("提交月度物化视图刷新", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.query("Doris 物化视图任务", """
select MvName, Status, CreateTime
from tasks('type'='mv')
where MvDatabaseName = 'dbt_demo_daily'
  and MvName = 'monthly_order_summary_mv'
order by CreateTime desc
limit 1
""")

### 2.6 校验完整数据链并查看最终结果

Verifier 等待异步刷新完成，同时检查每日数据、Table 的 Key/Partition/Bucket DDL，以及月度汇总值。

In [ ]:
runner.run_script("校验每日订单完整数据链", daily_demo_dir / "scripts/verify.sh")
runner.query("最终结果：月度订单汇总", """
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv
order by order_month
""")

## 3. Demo 2：客户地域分析

这个 Demo 展示两张 Doris Source 如何经过各自的 staging View，最后通过 `ref()` join 成州级客户指标。

```text
CUSTOMER_ADDRESSES + ORDERS
  └─ stg_customer_addresses / stg_orders（过滤默认地址和有效订单）
       └─ fct_state_customers（按州 join、聚合客户数/订单数/收入）
```

### 3.1 准备并查看两张源表

源数据放在两个 Doris Database：地址表有 4 条记录，其中 1 条不是默认收货地址；订单表有 4 条记录，其中 1 条是取消订单。

In [ ]:
geo_dir = runner.examples_root / "data-eng-bench-doris-demos/geographic"
runner.show_file("Fixture SQL", geo_dir / "scripts/setup.sql")
runner.run_sql_file("创建地域分析源表", geo_dir / "scripts/setup.sql")
runner.query("输入：客户地址", """
select address_id, customer_id, state_province, is_default_shipping
from dbt_demo_geographic_customer.CUSTOMER_ADDRESSES
order by address_id
""")
runner.query("输入：订单", """
select order_id, customer_id, grand_total, status
from dbt_demo_geographic_orders.ORDERS
order by order_id
""")

### 3.2 创建 staging View

两个 staging Model 分别使用 `source()` 读取源表：地址 Model 只保留默认收货地址，订单 Model 只保留 COMPLETED、DELIVERED、SHIPPED。

In [ ]:
runner.show_file("地址 staging Model", geo_dir / "models/stg_customer_addresses.sql")
runner.show_file("订单 staging Model", geo_dir / "models/stg_orders.sql")
runner.show_file("Source 声明", geo_dir / "models/sources.yml")
runner.run_dbt("检查地域 Demo 连接", geo_dir, "debug")
runner.run_dbt("创建两个 staging View", geo_dir, "run", "--select", "stg_customer_addresses", "stg_orders")
runner.query("中间结果：staging View", """
select 'address' as stage, cast(address_id as string) as record_id, state_province as value
from dbt_demo_geographic.stg_customer_addresses
union all
select 'order', cast(order_id as string), cast(grand_total as string)
from dbt_demo_geographic.stg_orders
order by stage, record_id
""")

### 3.3 通过 `ref()` 生成州级指标

事实 Model 不再直接读取源表，而是 join 两个 staging View：CA 有 2 个客户和 2 个有效订单，NY 有 1 个客户和 1 个有效订单。

In [ ]:
runner.show_file("州级指标 Model", geo_dir / "models/fct_state_customers.sql")
runner.run_dbt("创建 fct_state_customers", geo_dir, "run", "--select", "fct_state_customers")
runner.query("输出：州级客户与收入", """
select state_province, customer_count, order_count, total_revenue, avg_order_value
from dbt_demo_geographic.fct_state_customers
order by state_province
""")

### 3.4 执行 Data Test 并验证对象类型

Data Test 检查州名和客户数非空；verifier 还检查最终对象是 Table，两个中间对象是 View。

In [ ]:
runner.show_file("Data Test 定义", geo_dir / "models/geographic.yml")
runner.run_dbt("验证 fct_state_customers", geo_dir, "test", "--select", "fct_state_customers")
runner.run_script("校验地域 Demo", geo_dir / "scripts/verify.sh")
runner.query("最终对象类型", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_geographic'
order by table_name
""")

## 4. Demo 3：广告数据标准化和合并

这个 Demo 展示 CSV 如何通过 dbt Seed 进入 Doris，再经过字段标准化、`QUALIFY` 去重和 `union all` 合并。

```text
googleads.csv / metaads.csv / tiktokads.csv
  └─ dbt seed：googleads / metaads / tiktokads（Doris Table）
       └─ 3 个 staging View（字段统一 + QUALIFY 去重）
            └─ int__ads_unified（union all + source 标识）
```

### 4.1 准备 CSV 并执行 dbt Seed

Google 和 Meta 的 8 月 1 日有重复行，TikTok 没有重复行。先由 dbt seed 将三个 CSV 加载成 Doris 表。

In [ ]:
ads_dir = runner.examples_root / "data-eng-bench-doris-demos/consolidate"
runner.show_file("Fixture SQL", ads_dir / "scripts/setup.sql")
runner.show_file("Google CSV", ads_dir / "seeds/googleads.csv")
runner.show_file("Meta CSV", ads_dir / "seeds/metaads.csv")
runner.show_file("TikTok CSV", ads_dir / "seeds/tiktokads.csv")
runner.run_sql_file("创建广告 Demo 数据库", ads_dir / "scripts/setup.sql")
runner.run_dbt("检查广告 Demo 连接", ads_dir, "debug")
runner.run_dbt("安装 dbt_utils", ads_dir, "deps")
runner.run_dbt("加载三个广告 Seed", ads_dir, "seed", "--select", "googleads", "metaads", "tiktokads")
runner.query("Seed 输入行数", """
select 'googleads' as table_name, count(*) as table_rows from dbt_demo_consolidate.googleads
union all select 'metaads', count(*) from dbt_demo_consolidate.metaads
union all select 'tiktokads', count(*) from dbt_demo_consolidate.tiktokads
order by table_name
""")

### 4.2 标准化字段并去重

三个 staging Model 使用 `ref()` 读取 Seed：Meta 将 `views_1 + views_2` 合成 `views`，TikTok 将 `views_1` 映射为 `views`，然后用 `row_number()` + `QUALIFY` 删除重复记录。

In [ ]:
for model_name in ("googleads", "metaads", "tiktokads"):
    runner.show_file(f"{model_name} staging Model", ads_dir / f"models/stg__ads_{model_name}.sql")
runner.run_dbt("创建三个 staging View", ads_dir, "run", "--select", "stg__ads_googleads", "stg__ads_metaads", "stg__ads_tiktokads")
runner.query("中间结果：去重后的 staging", """
select 'google' as source, count(*) as rows_after_dedup from dbt_demo_consolidate.stg__ads_googleads
union all select 'meta', count(*) from dbt_demo_consolidate.stg__ads_metaads
union all select 'tiktok', count(*) from dbt_demo_consolidate.stg__ads_tiktokads
order by source
""")

### 4.3 合并三个渠道并执行唯一性测试

最终 Model 给每个 staging 增加 `source` 字段，用 `union all` 合并为一张表；`dbt_utils.unique_combination_of_columns` 检查 `source + ad_date` 不重复。

In [ ]:
runner.show_file("统一广告 Model", ads_dir / "models/int__ads_unified.sql")
runner.show_file("唯一性 Test 定义", ads_dir / "models/int__ads_unified.yml")
runner.run_dbt("创建统一广告表并测试", ads_dir, "build", "--select", "int__ads_unified")
runner.query("输出：统一广告明细", """
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date
""")

### 4.4 验证最终对象

Verifier 检查最终表有 6 行、日期非空，且三个 staging 对象是 View、统一对象是 Table。

In [ ]:
runner.run_script("校验广告合并 Demo", ads_dir / "scripts/verify.sh")
runner.query("最终对象类型", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_consolidate'
  and table_name in ('stg__ads_googleads', 'stg__ads_metaads', 'stg__ads_tiktokads', 'int__ads_unified')
order by table_name
""")

## 5. Demo 4：迟到订单 Incremental

这个 Demo 分阶段展示同一张订单源表发生变化后，dbt-doris 如何用 Unique Key + `merge` 更新当前版本。

```text
3 条初始订单事件
  └─ order_version_history（识别每个 order_id 的版本）
       └─ incremental_daily_sales（首次全量写入）
            └─ 新版本 101 + 新订单 104
                 └─ merge 后保留 4 个订单当前版本
```

### 5.1 准备并查看初始订单事件

源表按事件保存订单。订单 101、102、103 各有一个事件，后面会再写入订单 101 的新版本和订单 104。

In [ ]:
incremental_dir = runner.examples_root / "data-eng-bench-doris-demos/incremental"
runner.show_file("Fixture SQL", incremental_dir / "scripts/setup.sql")
runner.show_file("Source 声明", incremental_dir / "models/sources.yml")
runner.run_sql_file("创建 Incremental 源表", incremental_dir / "scripts/setup.sql")
runner.query("输入：初始订单事件", """
select event_id, order_id, customer_id, channel_id, grand_total, ordered_at, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 5.2 首次构建版本历史和 Incremental 表

`order_version_history` 用窗口函数给每个订单排序；`incremental_daily_sales` 只输出 `is_current = 1` 的版本，并配置 `unique_key='order_id'` 和 `incremental_strategy='merge'`。

In [ ]:
runner.show_file("版本历史 Model", incremental_dir / "models/order_version_history.sql")
runner.show_file("Incremental Model", incremental_dir / "models/incremental_daily_sales.sql")
runner.show_file("Incremental Test 定义", incremental_dir / "models/incremental.yml")
runner.run_dbt("检查 Incremental Demo 连接", incremental_dir, "debug")
runner.run_dbt("首次全量构建", incremental_dir, "build")
runner.query("中间结果：当前订单版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")

### 5.3 写入迟到版本和新订单

新事件把订单 101 的金额从 100.00 改为 125.00，并新增订单 104。此时目标表还没有变化，变化只发生在源事件表。

In [ ]:
late_events_sql = """
insert into dbt_demo_incremental_source.ORDERS values
    (4, 101, 1, 'web', 125.00, 'COMPLETED', '2026-08-01 09:00:00', '2026-08-05 09:00:00'),
    (5, 104, 3, 'mobile', 70.00, 'COMPLETED', '2026-08-01 12:00:00', '2026-08-05 10:00:00')
"""
runner.show_sql("迟到事件 SQL", late_events_sql)
runner.run_sql("写入迟到版本和新订单", late_events_sql)
runner.query("变更后的源事件", """
select event_id, order_id, grand_total, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 5.4 执行增量 merge

第二次 `dbt build` 重新计算依赖 Model，Incremental Model 通过 `order_id` 合并新旧记录：101 变为版本 2，104 成为版本 1。

In [ ]:
runner.run_dbt("执行增量 merge", incremental_dir, "build")
runner.query("merge 后的当前版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")
runner.query("merge 后的每日收入", """
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date
""")

### 5.5 再次运行确认幂等性

源表没有新增数据时再次执行相同的 build，结果仍应保持 4 个订单和 245.00 的 8 月 1 日收入。

In [ ]:
runner.run_dbt("重复构建 Incremental Model", incremental_dir, "build")
runner.run_script("校验 Incremental Demo", incremental_dir / "scripts/verify.sh")
runner.query("幂等运行后的结果", """
select count(*) as order_rows, count(distinct order_id) as distinct_orders,
       sum(case when order_date = '2026-08-01' then grand_total else 0 end) as aug_01_revenue
from dbt_demo_incremental.incremental_daily_sales
""")

## 6. Demo 5：客户 Snapshot

这个 Demo 分两轮展示 Snapshot：第一轮记录客户初始状态，第二轮在更新客户 1、删除客户 2 后写入新的 SCD Type 2 历史。

```text
2 条当前客户
  └─ stg_customers View
       └─ 第一次 snapshot：2 条有效历史
            └─ 更新客户 1 + 删除客户 2
                 └─ 第二次 snapshot：3 条历史、1 条当前客户
```

### 6.1 准备并查看当前客户源表

源表使用 Unique Key 保存客户当前状态，初始包含 Alice 和 Bob 两条记录。

In [ ]:
snapshot_dir = runner.examples_root / "data-eng-bench-doris-demos/snapshot"
runner.show_file("Fixture SQL", snapshot_dir / "scripts/setup.sql")
runner.show_file("Source 声明", snapshot_dir / "models/sources.yml")
runner.run_sql_file("创建客户源表", snapshot_dir / "scripts/setup.sql")
runner.query("输入：客户当前状态", """
select customer_id, customer_number, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 6.2 创建 staging View

`stg_customers` 通过 `source()` 读取当前客户表，为 Snapshot 提供稳定的上游节点。

In [ ]:
runner.show_file("客户 staging Model", snapshot_dir / "models/stg_customers.sql")
runner.run_dbt("检查 Snapshot Demo 连接", snapshot_dir, "debug")
runner.run_dbt("创建 stg_customers View", snapshot_dir, "run", "--select", "stg_customers")
runner.query("中间结果：staging 客户", """
select customer_id, customer_type, email
from dbt_demo_snapshot.stg_customers
order by customer_id
""")

### 6.3 执行第一次 Snapshot

Snapshot 使用 `check` 策略监控邮箱、客户类型等字段，并开启 `invalidate_hard_deletes`。第一次执行会为两个客户各创建一个有效版本。

In [ ]:
runner.show_file("Snapshot 定义", snapshot_dir / "snapshots/customer_snapshot.sql")
runner.run_dbt("记录客户初始版本", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.query("第一次 Snapshot：2 条有效历史", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")

### 6.4 从 Snapshot 生成当前客户维表

维表只保留 `dbt_valid_to is null` 的当前版本，并统计每个客户共有多少历史版本。第一次构建时两个客户都只有一个版本。

In [ ]:
runner.show_file("当前客户维表 Model", snapshot_dir / "models/dim_customer_current.sql")
runner.show_file("Data Test 定义", snapshot_dir / "models/snapshot.yml")
runner.run_dbt("创建当前客户维表", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("测试当前客户维表", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("第一次维表结果", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 6.5 修改和删除源客户

将客户 1 的邮箱和类型更新，并从当前源表删除客户 2。此时 Snapshot 历史尚未变化。

In [ ]:
customer_changes_sql = """
update dbt_demo_snapshot_source.CUSTOMERS
set email = 'alice.new@example.com', customer_type = 'INDIVIDUAL_PLUS'
where customer_id = 1;
delete from dbt_demo_snapshot_source.CUSTOMERS where customer_id = 2
"""
runner.show_sql("客户变更 SQL", customer_changes_sql)
runner.run_sql("更新 Alice 并删除 Bob", customer_changes_sql)
runner.query("变更后的当前源表", """
select customer_id, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 6.6 执行第二次 Snapshot 并刷新维表

第二次 Snapshot 关闭 Alice 的旧版本并创建新版本，同时关闭已从源表删除的 Bob。随后刷新维表，只剩 Alice 的当前版本。

In [ ]:
runner.run_dbt("记录客户变更版本", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.run_dbt("刷新当前客户维表", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("测试刷新后的客户维表", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("第二次 Snapshot：3 条历史", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")
runner.query("刷新后的当前客户维表", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 6.7 校验 SCD Type 2 历史

Verifier 检查 Alice 有一个关闭版本和一个当前版本，Bob 只有关闭版本，当前维表只包含 Alice。

In [ ]:
runner.run_script("校验 Snapshot Demo", snapshot_dir / "scripts/verify.sh")

## 完成

5 个 Demo 的全部步骤显示绿色“执行通过”，表示对应源数据、dbt node、Doris 中间对象、Data Test 和 verifier 均已通过。需要排查编译 SQL 或执行细节时，展开每个结果下方的“查看完整运行日志”。